In [12]:
import argparse
import glob
import os
from dataclasses import dataclass
from typing import Optional, Tuple, Union

import numpy as np
import pandas as pd


# -------------------------------
# Physical constants and units
# -------------------------------
EV_TO_GEV = 1.0e-9
M_TO_GEV_INV = 5.0677e15
GEV_PER_CM3_TO_GEV4 = 1.0 / 1.3e41
ALPHA_EM = 1.0 / 137.035999084
E_CHARGE = np.sqrt(4.0 * np.pi * ALPHA_EM)

RHO_DM_GEV_PER_CM3 = 0.45
RHO_DM = RHO_DM_GEV_PER_CM3 * GEV_PER_CM3_TO_GEV4
C_MDM_TE011 = 0.22


@dataclass(frozen=True)
class CavityConfig:
    radius_m: Optional[float]
    c_axion: Optional[Union[float, Tuple[float, float]]]
    label: str
    note: str = ""

    @property
    def supports_simple_recast(self) -> bool:
        return self.radius_m is not None and self.c_axion is not None

    @property
    def radius_gev_inv(self) -> float:
        if self.radius_m is None:
            raise ValueError(f"{self.label} does not define a simple cavity radius")
        return self.radius_m * M_TO_GEV_INV

    @property
    def c_axion_central(self) -> float:
        if self.c_axion is None:
            raise ValueError(f"{self.label} does not define a simple axion form factor")
        if isinstance(self.c_axion, tuple):
            c_min, c_max = self.c_axion
            return np.sqrt(c_min * c_max)
        return self.c_axion


# The keys are the usual AxionLimits file stems.
# ADMX, HAYSTAC, and QUAX values use the published effective TM010-like form
# factor for the corresponding tuned cavity mode. Sidecar values span the
# reported form-factor range, so the script writes an additional range file for
# those inputs. QUAX TM030 dielectric-cavity papers quote an effective VC030
# product; those entries are included as named configurations but skipped by the
# simple TE011-cylinder recast until a dedicated mDM overlap is supplied.
CAVITY_CONFIGS = {
    "ADMX": CavityConfig(0.419 / 2.0, 0.69, "legacy ADMX/SQUID"),
    "ADMX2018": CavityConfig(0.21, 0.40, "ADMX Run 1A"),
    "ADMX2019_1": CavityConfig(0.21, 0.40, "ADMX Run 1B, lower band"),
    "ADMX2019_2": CavityConfig(0.21, 0.40, "ADMX Run 1B, upper band"),
    "ADMX2021": CavityConfig(0.21, 0.40, "ADMX Run 1C"),
    "ADMX2024": CavityConfig(0.21, 0.40, "ADMX 2022 scan"),
    "ADMX2025": CavityConfig(0.2095, 0.40, "ADMX 1.1--1.3 GHz"),
    "ADMX_Sidecar": CavityConfig(0.032, (0.04, 0.61), "ADMX Sidecar A--C"),
    "ADMX_Sidecar_AC": CavityConfig(0.032, (0.04, 0.49), "ADMX Sidecar A,C"),
    "HAYSTAC_PhaseI": CavityConfig(0.051, 0.50, "HAYSTAC Phase I"),
    "HAYSTAC_PhaseII_ab": CavityConfig(0.051, 0.43, "HAYSTAC Phase IIa/IIb"),
    "HAYSTAC_PhaseII_cd": CavityConfig(0.051, 0.43, "HAYSTAC Phase IIc/IId"),
    # Optional aliases for locally split files.
    "HAYSTAC_PhaseIIa": CavityConfig(0.051, 0.43, "HAYSTAC Phase IIa"),
    "HAYSTAC_PhaseIIb": CavityConfig(0.051, 0.43, "HAYSTAC Phase IIb"),
    "HAYSTAC_PhaseIIc": CavityConfig(0.051, 0.43, "HAYSTAC Phase IIc"),
    "HAYSTAC_PhaseIId": CavityConfig(0.051, 0.43, "HAYSTAC Phase IId"),
    "QUAX": CavityConfig(0.0261 / 2.0, 0.589, "QUAX-aγ SCC, 2019"),
    "QUAX2": CavityConfig(0.011, 0.69, "QUAX-aγ JPA, 2021"),
    "QUAX3": CavityConfig(None, None, "QUAX high-Q dielectric cavity, 2022", "Uses published VC030 = 0.034 L; mode-specific mDM overlap required.",),
    "QUAX4": CavityConfig(None, None, "QUAX dielectric cavity with TWPA, 2023", "Uses published VC030 = 0.034 L; mode-specific mDM overlap required.", ),
    "QUAX5": CavityConfig(0.0135, 0.69, "QUAX-LNF tunable haloscope, 2024"),
}


def epsilon_from_axion_limit(
    m_a_eV: pd.Series,
    g_agamma_GeV_inv: pd.Series,
    config: CavityConfig,
    c_axion: float,
) -> pd.Series:
    """Convert an axion-photon limit into a fractional millicharge limit."""
    m_phi_eV = m_a_eV / 2.0
    m_phi_GeV = m_phi_eV * EV_TO_GEV
    m_a_GeV = m_a_eV * EV_TO_GEV
    c_ratio = c_axion / C_MDM_TE011

    inside = (
        g_agamma_GeV_inv**2
        * 8.0
        * m_phi_GeV**5
        * c_ratio
        / (config.radius_gev_inv**2 * RHO_DM * m_a_GeV)
    )
    return np.power(inside, 0.25) / E_CHARGE


def read_axion_limit(path: str) -> pd.DataFrame:
    return pd.read_csv(
        path,
        comment="#",
        sep=r"\s+",
        header=None,
        names=["m_a_eV", "g_agamma_GeV_inv"],
    ).dropna()


def write_two_column_output(path: str, data: pd.DataFrame, config: CavityConfig) -> None:
    with open(path, "w", encoding="utf-8") as output:
        output.write("# Rescaled haloscope axion-photon limit to millicharged dark matter\n")
        output.write(f"# Source/run: {config.label}\n")
        output.write("# m_phi = m_a / 2 for the same cavity frequency\n")
        output.write(f"# radius_m = {config.radius_m:g}\n")
        output.write(f"# C_axion = {config.c_axion_central:g}\n")
        output.write(f"# C_mDM_TE011 = {C_MDM_TE011:g}\n")
        output.write("# m_phi [eV]    epsilon = e_m/e\n")
        data.to_csv(output, sep="\t", index=False, header=False)


def write_range_output(path: str, data: pd.DataFrame, config: CavityConfig) -> None:
    with open(path, "w", encoding="utf-8") as output:
        output.write("# Rescaled ADMX Sidecar limit with reported C_axion range\n")
        output.write(f"# Source/run: {config.label}\n")
        output.write("# m_phi = m_a / 2 for the same cavity frequency\n")
        output.write(f"# radius_m = {config.radius_m:g}\n")
        output.write(f"# C_axion range = {config.c_axion[0]:g}--{config.c_axion[1]:g}\n")
        output.write(f"# C_mDM_TE011 = {C_MDM_TE011:g}\n")
        output.write("# m_phi [eV]    epsilon(C_min)    epsilon(C_geom)    epsilon(C_max)\n")
        data.to_csv(output, sep="\t", index=False, header=False)


def process_file(file_path: str, output_dir: str) -> bool:
    filename = os.path.basename(file_path)
    stem, ext = os.path.splitext(filename)
    config = CAVITY_CONFIGS.get(stem)

    if config is None:
        print(f"Skipping {filename}: no cavity configuration defined")
        return False

    if not config.supports_simple_recast:
        suffix = f": {config.note}" if config.note else ""
        print(f"Skipping {filename}: {config.label} needs a dedicated recast{suffix}")
        return False

    print(f"Processing {filename} with {config.label} parameters ...")
    df = read_axion_limit(file_path)
    df["m_phi_eV"] = df["m_a_eV"] / 2.0
    df["epsilon"] = epsilon_from_axion_limit(
        df["m_a_eV"],
        df["g_agamma_GeV_inv"],
        config,
        config.c_axion_central,
    )

    output_path = os.path.join(output_dir, f"{stem}_rescaled{ext}")
    write_two_column_output(output_path, df[["m_phi_eV", "epsilon"]], config)
    print(f"  -> Saved to {output_path}")

    if isinstance(config.c_axion, tuple):
        c_min, c_max = config.c_axion
        range_df = pd.DataFrame(
            {
                "m_phi_eV": df["m_phi_eV"],
                "epsilon_cmin": epsilon_from_axion_limit(
                    df["m_a_eV"], df["g_agamma_GeV_inv"], config, c_min
                ),
                "epsilon_cgeom": df["epsilon"],
                "epsilon_cmax": epsilon_from_axion_limit(
                    df["m_a_eV"], df["g_agamma_GeV_inv"], config, c_max
                ),
            }
        )
        range_path = os.path.join(output_dir, f"{stem}_rescaled_Crange{ext}")
        write_range_output(range_path, range_df, config)
        print(f"  -> Saved to {range_path}")

    return True


def main() -> None:
    parser = argparse.ArgumentParser(
        description="Rescale ADMX/HAYSTAC/QUAX AxionLimits data to millicharged dark matter."
    )
    parser.add_argument("--input-dir", default="AxionPhoton")
    parser.add_argument("--output-dir", default="Rescale")
    args, unknown_args = parser.parse_known_args()
    if unknown_args:
        print(f"Ignoring unknown arguments: {' '.join(unknown_args)}")

    os.makedirs(args.output_dir, exist_ok=True)
    paths = sorted(glob.glob(os.path.join(args.input_dir, "*.txt")))
    if not paths:
        print(f"No .txt files found in {args.input_dir}")
        return

    processed = sum(process_file(path, args.output_dir) for path in paths)
    print(f"Done. Processed {processed} supported haloscope file(s).")


if __name__ == "__main__":
    main()

Ignoring unknown arguments: -f C:\Users\usuario\AppData\Roaming\jupyter\runtime\kernel-72a9c280-cf14-462f-a376-a4e3edfa340a.json
Skipping ABRACADABRA.txt: no cavity configuration defined
Skipping ABRACADABRA_run2.txt: no cavity configuration defined
Skipping ADBC.txt: no cavity configuration defined
Processing ADMX.txt with legacy ADMX/SQUID parameters ...
  -> Saved to Rescale\ADMX_rescaled.txt
Processing ADMX2018.txt with ADMX Run 1A parameters ...
  -> Saved to Rescale\ADMX2018_rescaled.txt
Processing ADMX2019_1.txt with ADMX Run 1B, lower band parameters ...
  -> Saved to Rescale\ADMX2019_1_rescaled.txt
Processing ADMX2019_2.txt with ADMX Run 1B, upper band parameters ...
  -> Saved to Rescale\ADMX2019_2_rescaled.txt
Processing ADMX2021.txt with ADMX Run 1C parameters ...
  -> Saved to Rescale\ADMX2021_rescaled.txt
Processing ADMX2024.txt with ADMX 2022 scan parameters ...
  -> Saved to Rescale\ADMX2024_rescaled.txt
Processing ADMX2025.txt with ADMX 1.1--1.3 GHz parameters ...
  ->